# Step 10 — Aftermath: Plan vs. Reality
**Tough Talks · Phase 4**

Goal: prove Gemma 4 E2B (text-only) can compare the **plan** (a pre-mortem produced before the conversation, Step 8) against the **reality** (the actual practice transcript, Step 7) and return a structured plan-vs-reality read.

The output is one structured JSON object matching `data/schemas/aftermath.schema.json`:

- `goal_outcome` — `{status, summary}`. Did the user get what they came for? Enum: `achieved | partial | not_achieved`.
- `scenario_outcomes[]` — exactly three entries, one per predicted pre-mortem scenario, in the same order. Each carries `scenario_id`, `title` and `predicted_resistance_type` (copied verbatim from the pre-mortem in code, NOT trusted to the model), `match_quality` enum (`direct_hit | partial | did_not_occur`), derived `materialized` boolean, `evidence`, and `notes`.
- `unforeseen_moments[]` — `{kind, [turn], description}`. Risks or opportunities the pre-mortem didn't predict but actually showed up. Empty array is valid.
- `prediction_accuracy` — number in `[0, 1]`. How well the pre-mortem matched reality overall.
- `next_round_focus` — single sentence pointing at what the next practice round should target.

**Architecture** — same one-shot hybrid-runtime pattern as Steps 8 / 9:
- Runtime in `backend/core/_runtime/aftermath.py`. Notebook is a thin driver.
- Single prompt-based JSON call (analytical, not multi-turn).
- Two-shot retry: greedy first, then a single light-sampling pass (`temperature=0.3, top_p=0.9, top_k=64`) on JSON / validation failure.
- System-contract enforcement in code (defence-in-depth, same shape as Step 04's `whisper_prompt=None for other`, Step 07's `persona_name` forcing, Step 08's `scenario_id` forcing, Step 09's `_APOLOGY_CUE_RE`):
  - Exactly 3 `scenario_outcomes`, in pre-mortem order.
  - `scenario_id` forced to `index + 1`; `title` and `predicted_resistance_type` copied **from the pre-mortem input**, not from model output — the model frequently paraphrases titles or re-types enums.
  - `match_quality` enum-normalised (`exact` → `direct_hit`, `missed` → `did_not_occur`, etc.).
  - `materialized` derived from `match_quality` in code — `direct_hit` / `partial` → `true`, `did_not_occur` → `false`. The model's boolean is discarded when it disagrees.
  - `did_not_occur` forces `evidence = ""` (the prompt explicitly forbids invented evidence for a missed prediction). `direct_hit` / `partial` with empty evidence is silently downgraded to `did_not_occur`.
  - `unforeseen_moments[].kind` enum-normalised; optional `turn` clamped to `[1, user_turn_count]`; malformed entries dropped silently.
  - `prediction_accuracy` clamped to `[0, 1]`; `next_round_focus` required and non-empty.
- `enable_thinking` is exposed as a knob, defaulting to `False`. The final A/B cell is the **third Phase 4 analytical-task datapoint** for `[[hypothesis-persona-thinking-helps]]` — premortem (N=2) and debrief (N=1) both showed thinking-on helping on cross-field consistency. Aftermath has the most cross-field structure yet: predicted resistance vs. actual transcript content vs. user response, all in one payload.

**What "done" looks like for this step**
1. Text-only Gemma 4 loads (`AutoModelForCausalLM` via `LoadConfig(multimodal=False)`).
2. `generate_aftermath()` returns a schema-conforming dict on the hardcoded Step-7 transcript and a representative Step-8 pre-mortem.
3. `scenario_outcomes` has exactly 3 entries with `scenario_id` 1 / 2 / 3 in order.
4. Every scenario's `title` and `predicted_resistance_type` match the pre-mortem input (defence-in-depth check — runtime forces this regardless of model output).
5. `materialized` is consistent with `match_quality` on every entry.
6. `did_not_occur` scenarios have empty `evidence`; `direct_hit` / `partial` scenarios have non-empty `evidence`.
7. `prediction_accuracy ∈ [0, 1]` and `next_round_focus` is non-empty.
8. The A/B cell renders both `enable_thinking=False` and `enable_thinking=True` outputs side-by-side.

**Note on inputs.** Jamie's PersonVault profile, the user's TalkDNA, the 5-turn practice transcript, the pre-mortem, and the post-round debrief are all hardcoded inline (matching Step 06 v2 / Step 05 v2 / Step 07 Run-2 / Step 08 thinking-on / Step 09 thinking-on outputs respectively) so this notebook doesn't re-pay Steps 05 / 06 / 07 / 08 / 09's combined ~45-minute LLM cost on every run. In production these come from `analyze_person_vault()`, `analyze_talk_dna()`, `run_practice_conversation()`, `generate_premortem()`, and `generate_debrief()`.

In [1]:
# ── 0. Install / upgrade dependencies ────────────────
# Text-only path — no audio libs required. Same rule as Steps 05 / 06 /
# 07 / 08 / 09: bump only transformers + accelerate on Colab / Kaggle
# (bumping torch breaks the pre-installed torchvision / CUDA pairing).
# After this first run, RESTART THE KERNEL before continuing if you
# actually upgraded transformers — the already-imported version won't
# pick up the change.

!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 105.3 MB/s eta 0:00:0000:010:01


In [2]:
# ── 1. Locate (or fetch) the repo, put it on sys.path ───────────
# Same shim as Steps 01–09 — auto-clones / refreshes on Colab / Kaggle
# and clears any cached `backend.*` modules so the imports below pick
# up the freshly-pulled code instead of whatever this kernel imported
# earlier in the session.

import os, pathlib, subprocess, sys

REPO_URL  = "https://github.com/EhsanFarazmand/tough_talks.git"
REPO_NAME = "tough_talks"

def _looks_like_repo(p: pathlib.Path) -> bool:
    return (p / "backend" / "core" / "_runtime").is_dir()

def _scan_for_repo() -> pathlib.Path | None:
    cwd = pathlib.Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if _looks_like_repo(parent):
            return parent
    for base in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")):
        candidate = base / REPO_NAME
        if _looks_like_repo(candidate):
            return candidate
    return None

def _refresh(target: pathlib.Path) -> None:
    if not (target / ".git").is_dir():
        return
    print(f"Refreshing {target} from origin")
    subprocess.run(["git", "-C", str(target), "fetch", "--depth", "1", "origin"],
                   capture_output=True, check=False)
    subprocess.run(["git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"],
                   capture_output=True, check=False)

REPO_ROOT = _scan_for_repo()
if REPO_ROOT is None:
    base = next((b for b in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")) if b.is_dir()),
                pathlib.Path.cwd())
    target = base / REPO_NAME
    print(f"Cloning {REPO_URL} -> {target}")
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed:\n" + result.stderr)
    REPO_ROOT = target
else:
    _refresh(REPO_ROOT)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

_stale = [m for m in list(sys.modules) if m == "backend" or m.startswith("backend.")]
for _m in _stale:
    del sys.modules[_m]
if _stale:
    print(f"Cleared {len(_stale)} cached backend.* module(s) from sys.modules")

print(f"Repo root: {REPO_ROOT}")

Cloning https://github.com/EhsanFarazmand/tough_talks.git -> /content/tough_talks
Repo root: /content/tough_talks


In [3]:
# ── 2. Imports ──────────────────────────────
import json
from pathlib import Path

import torch

from backend.core._runtime import (
    ALLOWED_GOAL_STATUSES,
    ALLOWED_MATCH_QUALITIES,
    ALLOWED_UNFORESEEN_KINDS,
    AftermathConfig,
    AftermathError,
    DEFAULT_MODEL_ID,
    LoadConfig,
    count_user_turns,
    format_debrief_block,
    format_persona_profile_block,
    format_practice_transcript,
    format_premortem_block,
    generate_aftermath,
    load_model,
)

In [4]:
# ── 3. Configuration ──────────────────────────

MODEL_ID    = DEFAULT_MODEL_ID                # google/gemma-4-E2B-it
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
SCHEMA_PATH = Path(REPO_ROOT) / "data" / "schemas" / "aftermath.schema.json"

# PersonVault profile for Jamie — same v2 dict as Steps 07 / 08 / 09.
JAMIE_PROFILE = {
    "person_id": "person_cf528d57b789",
    "name": "Jamie",
    "relationship_type": "colleague",
    "version": 2,
    "conversation_count": 2,
    "profile": {
        "communication_style": "defensive",
        "emotional_triggers": [
            "citing past commitments",
            "suggesting escalation when blockers are still open",
            "implying missed communication is one-sided",
            "setting hard deadlines without acknowledging blockers",
        ],
        "de_escalation_keys": [
            "explicitly disowning blame",
            "reframing as joint problem-solving",
            "acknowledging the tightness of the timeline",
            "proposing a concrete next-step the user will own",
        ],
        "common_deflections": [
            "I told you",
            "Don't blame me",
            "Don't put this on me",
            "You weren't there",
            "staging tables aren't done",
        ],
        "responds_best_to": (
            "Clear, actionable next steps tied to specific milestones. "
            "Responds well when the user accepts responsibility for "
            "communication gaps."
        ),
    },
}

# The user goal that drove Step 07's simulation, Step 08's pre-mortem,
# and Step 09's debrief.
USER_GOAL = (
    "Get Jamie to commit to delivering the staging-tables data by "
    "Wednesday EOD and to acknowledge that the escalation last week "
    "needed to land more clearly — not just be sent."
)

# Representative Step 08 pre-mortem output. Three failure scenarios
# grounded in Jamie's profile — one `deflect`, one `counter_attack`,
# one `guilt_trip`. Matches the shape `generate_premortem()` returns,
# inlined so this notebook doesn't re-pay Step 08's LLM cost.
PREMORTEM = {
    "goal": USER_GOAL,
    "failure_scenarios": [
        {
            "scenario_id": 1,
            "title": "The 'I told you' blame redirect",
            "description": (
                "You bring up last week's missed deadline and Jamie pivots straight "
                "to 'I told you the staging tables weren't done.' The conversation "
                "slides into a debate over who knew what on which day, and the "
                "Wednesday EOD ask never lands."
            ),
            "likely_trigger": "User cites the missed Tuesday deadline directly",
            "destabilization_risk": 0.55,
            "simulation_parameters": {
                "resistance_type": "deflect",
                "escalation_ceiling": 0.65,
                "opening_move": (
                    "I told you the staging tables weren't done. "
                    "That's been on the data team since last Wednesday."
                ),
            },
        },
        {
            "scenario_id": 2,
            "title": "Counter-attack on the new deadline",
            "description": (
                "You set Wednesday EOD as the new deadline and Jamie pushes back "
                "by reframing your ask as overreach — demanding you spell out "
                "every blocker yourself before she'll commit to anything. The "
                "burden shifts back to you and Wednesday quietly slips."
            ),
            "likely_trigger": "User sets a hard date without naming the blockers Jamie sees",
            "destabilization_risk": 0.75,
            "simulation_parameters": {
                "resistance_type": "counter_attack",
                "escalation_ceiling": 0.85,
                "opening_move": (
                    "Wednesday EOD is unrealistic unless you can tell me exactly "
                    "what blockers you're anticipating. Don't just set a deadline "
                    "without outlining the path."
                ),
            },
        },
        {
            "scenario_id": 3,
            "title": "'Don't put this on me' guilt-trip",
            "description": (
                "You raise the missed-escalation point and Jamie flips it into a "
                "complaint about being unfairly blamed — 'don't put this on me, "
                "the process is broken'. You back off the original ask to soothe "
                "her, and the commitment never lands."
            ),
            "likely_trigger": "User implies the escalation failure is one-sided",
            "destabilization_risk": 0.60,
            "simulation_parameters": {
                "resistance_type": "guilt_trip",
                "escalation_ceiling": 0.70,
                "opening_move": (
                    "Don't put this on me. I escalated it twice last week — "
                    "if the communication isn't clear, that's on the process, not me."
                ),
            },
        },
    ],
    "premortem_id": "premortem_demo_8a1b2c3d4e5f",
    "generated_at": "2026-05-14T15:00:00+00:00",
}

# Step 09 post-round debrief output — matches the thinking-on payload
# from `step09_debrief.ipynb`'s final handoff cell. Optional input;
# the aftermath also works without it.
DEBRIEF = {
    "ground_lost": [],
    "over_apologies": [],
    "missed_openings": [],
    "wins": [
        {
            "turn": 2,
            "description": (
                "You successfully moved the focus from past blame to a concrete, "
                "shared commitment by setting a specific deadline: 'Can we agree "
                "the staging data lands by Wednesday EOD this time?'"
            ),
        },
        {
            "turn": 3,
            "description": (
                "You used a strong de-escalation move by reframing the issue as "
                "a joint problem, which directly addressed Jamie's defensive stance."
            ),
        },
        {
            "turn": 4,
            "description": (
                "You proposed a clear, actionable next step — the daily check-in — "
                "and put the responsibility for unblocking squarely on yourself."
            ),
        },
    ],
    "one_fix_next_time": (
        "When you propose a solution, immediately follow it up with the specific "
        "commitment you need from the other person to make that solution happen."
    ),
    "debrief_id": "debrief_demo_6da7108613b3",
    "generated_at": "2026-05-14T16:20:00+00:00",
}

# Five-turn practice transcript, lifted verbatim from Step 07's Run-2
# results table. Same shape Step 09 used.
PRACTICE_TRANSCRIPT = [
    {
        "speaker": "user",
        "text": (
            "Hey Jamie, I want to talk about what happened with the report "
            "last week. The fallout from missing that deadline is still on "
            "us, and I'd like to make sure it doesn't happen again."
        ),
    },
    {
        "speaker": "persona",
        "persona_name": "Jamie",
        "reply": (
            "I told you we were short on time. It's not a personal failing, "
            "it's just how things run. What exactly do you want me to do "
            "differently this time?"
        ),
        "resistance_type": "deflect",
        "escalation_level": 0.60,
    },
    {
        "speaker": "user",
        "text": (
            "I hear that the data team hadn't delivered, but the report "
            "date matters more than who knew what on which day. Can we "
            "agree the staging data lands by Wednesday EOD this time?"
        ),
    },
    {
        "speaker": "persona",
        "persona_name": "Jamie",
        "reply": (
            "Wednesday EOD is tight. I can try to push it, but I need to "
            "know exactly what blockers you're anticipating so I can manage "
            "expectations properly. Don't just set a deadline without "
            "outlining the path to get there."
        ),
        "resistance_type": "counter_attack",
        "escalation_level": 0.75,
    },
    {
        "speaker": "user",
        "text": (
            "I'm not saying you didn't escalate. I'm saying the escalation "
            "has to land — if I miss your emails, that's still our problem "
            "to solve together, not yours alone."
        ),
    },
    {
        "speaker": "persona",
        "persona_name": "Jamie",
        "reply": (
            "I told you, don't put this on me. If the communication isn't "
            "clear, that's on the process, not just me. I'll look at the "
            "timeline again, but I can't guarantee Wednesday EOD without a "
            "concrete commitment from your side on what you can handle."
        ),
        "resistance_type": "guilt_trip",
        "escalation_level": 0.70,
    },
    {
        "speaker": "user",
        "text": (
            "What if we set up a five-minute Slack check-in each morning "
            "until the staging tables are done? You flag what's blocked, I "
            "unblock it that day. That way nothing has to escalate at all."
        ),
    },
    {
        "speaker": "persona",
        "persona_name": "Jamie",
        "reply": (
            "A daily check-in sounds like a lot of overhead right now. If "
            "you're going to own the unblocking, then fine. Send me the "
            "specific blockers you see by the end of today, and I'll see "
            "what I can manage on my end."
        ),
        "resistance_type": "deflect",
        "escalation_level": 0.50,
    },
    {
        "speaker": "user",
        "text": (
            "Thank you, that means a lot. And next time something blocks "
            "the report, can you ping me directly instead of email? I'll "
            "do the same when I'm the one missing things."
        ),
    },
    {
        "speaker": "persona",
        "persona_name": "Jamie",
        "reply": (
            "I can do that. Just make sure those pings are actionable, not "
            "just general updates. I'll keep an eye out for them."
        ),
        "resistance_type": "concede",
        "escalation_level": 0.40,
    },
]

N_USER_TURNS = count_user_turns(PRACTICE_TRANSCRIPT)

print(f"Model              : {MODEL_ID}")
print(f"Device             : {DEVICE}")
print(f"Persona            : {JAMIE_PROFILE['name']} ({JAMIE_PROFILE['relationship_type']})")
print(f"Transcript turns   : {len(PRACTICE_TRANSCRIPT)} total / {N_USER_TURNS} USER")
print(f"Pre-mortem scenarios: {len(PREMORTEM['failure_scenarios'])} (predicted resistance: "
      f"{[s['simulation_parameters']['resistance_type'] for s in PREMORTEM['failure_scenarios']]})")
print(f"Debrief             : {len(DEBRIEF['wins'])} wins, {len(DEBRIEF['ground_lost'])} losses")

Model              : google/gemma-4-E2B-it
Device             : cuda
Persona            : Jamie (colleague)
Transcript turns   : 10 total / 5 USER
Pre-mortem scenarios: 3 (predicted resistance: ['deflect', 'counter_attack', 'guilt_trip'])
Debrief             : 3 wins, 0 losses


In [5]:
# ── 4. Preview the rendered context blocks (no model needed) ──────────
# All four rendered blocks are what the runtime feeds into the prompt.
# Rendering them here is purely diagnostic — it lets us verify the
# inputs are well-formed before paying for the model load.

print("=" * 76)
print("PERSON PROFILE BLOCK (Jamie)")
print("=" * 76)
print(format_persona_profile_block(JAMIE_PROFILE))

print()
print("=" * 76)
print("PRE-MORTEM BLOCK (3 predicted failure scenarios)")
print("=" * 76)
print(format_premortem_block(PREMORTEM))

print()
print("=" * 76)
print("DEBRIEF BLOCK (post-round coaching summary)")
print("=" * 76)
print(format_debrief_block(DEBRIEF))

print()
print("=" * 76)
print("PRACTICE TRANSCRIPT (5 user turns, 5 Jamie turns)")
print("=" * 76)
print(format_practice_transcript(PRACTICE_TRANSCRIPT))

PERSON PROFILE BLOCK (Jamie)
- communication_style: defensive
- emotional_triggers (USER-side cues that escalate / make you defensive): ['citing past commitments', 'suggesting escalation when blockers are still open', 'implying missed communication is one-sided', 'setting hard deadlines without acknowledging blockers']
- de_escalation_keys (USER-side moves that calm / open you up): ['explicitly disowning blame', 'reframing as joint problem-solving', 'acknowledging the tightness of the timeline', 'proposing a concrete next-step the user will own']
- common_deflections (YOUR habitual evasion phrases): ['I told you', "Don't blame me", "Don't put this on me", "You weren't there", "staging tables aren't done"]
- responds_best_to: Clear, actionable next steps tied to specific milestones. Responds well when the user accepts responsibility for communication gaps.

PRE-MORTEM BLOCK (3 predicted failure scenarios)
S1. The 'I told you' blame redirect — predicted resistance_type=deflect, escalatio

In [6]:
# ── 5. Load the text-only processor + model ────────────────
# Aftermath reasons over WORDS — same rationale as TalkDNA / PersonVault
# / persona-sim / premortem / debrief. The `multimodal=False` (default)
# path loads `AutoModelForCausalLM`, which is lighter on VRAM and
# slightly faster than the multimodal class. Live Mode (Phase 5+) will
# reuse this same loaded model across components, so the load cost is
# amortised.

processor, model = load_model(LoadConfig(model_id=MODEL_ID))
n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded ({n_params:.1f}B parameters, on {model.device})")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded (5.1B parameters, on cuda:0)


In [7]:
# ── 6. Generate the aftermath (greedy, thinking=False) ─────────────
# Single prompt-based JSON call — no rolling history (the opposite of
# Step 7's persona simulator, same shape as Steps 08 / 09). The runtime
# threads the user goal, person profile, pre-mortem, debrief, and
# rendered transcript into the `data/prompts/aftermath.md` template,
# calls `chat()` greedy, parses the JSON, validates / coerces against
# the schema, and returns a schema-conforming dict.

cfg = AftermathConfig(
    premortem=PREMORTEM,
    transcript=PRACTICE_TRANSCRIPT,
    user_goal=USER_GOAL,
    person_profile=JAMIE_PROFILE,
    debrief=DEBRIEF,
    enable_thinking=False,
)

try:
    aftermath = generate_aftermath(processor, model, cfg=cfg)
except AftermathError as exc:
    print(f"FAILED: {exc}")
    for attempt in getattr(exc, "attempts", []):
        print(f"  attempt ({attempt['sampling']}): {attempt['error']}")
        print(f"  raw_head: {attempt['raw_head']!r}")
    raise

print("=" * 76)
print("GOAL OUTCOME")
print("=" * 76)
print(f"  status : {aftermath['goal_outcome']['status']}")
print(f"  summary: {aftermath['goal_outcome']['summary']}")

print()
print("=" * 76)
print("SCENARIO OUTCOMES (3 — in pre-mortem order)")
print("=" * 76)
for outcome in aftermath["scenario_outcomes"]:
    sid = outcome["scenario_id"]
    print(f"\nS{sid}. {outcome['title']}")
    print(f"    predicted_resistance_type: {outcome['predicted_resistance_type']}")
    print(f"    match_quality            : {outcome['match_quality']}")
    print(f"    materialized             : {outcome['materialized']}")
    print(f"    evidence                 : {outcome['evidence']!r}")
    print(f"    notes                    : {outcome['notes']}")

print()
print("=" * 76)
print(f"UNFORESEEN MOMENTS ({len(aftermath['unforeseen_moments'])})")
print("=" * 76)
if not aftermath["unforeseen_moments"]:
    print("  (none)")
for moment in aftermath["unforeseen_moments"]:
    turn_str = f" @USER {moment['turn']}" if "turn" in moment else ""
    print(f"  [{moment['kind']}{turn_str}] {moment['description']}")

print()
print(f"prediction_accuracy: {aftermath['prediction_accuracy']:.2f}")
print(f"next_round_focus   : {aftermath['next_round_focus']}")

GOAL OUTCOME
  status : partial
  summary: You got some good movement on the next steps, but Jamie still pushed back on the hard deadline by asking for more detail, meaning the commitment wasn't fully secured yet.

SCENARIO OUTCOMES (3 — in pre-mortem order)

S1. The 'I told you' blame redirect
    predicted_resistance_type: deflect
    match_quality            : direct_hit
    materialized             : True
    evidence                 : "Jamie (resistance=deflect, escalation=0.60) I told you we were short on time. It's not a personal failing, it's just how things run."
    notes                    : This played out exactly as predicted. Jamie immediately used the 'I told you' deflection when you brought up the missed deadline, shifting the focus away from the issue itself.

S2. Counter-attack on the new deadline
    predicted_resistance_type: counter_attack
    match_quality            : partial
    materialized             : True
    evidence                 : "Jamie (resistance=co

In [8]:
# ── 7. Schema validation + results table ─────────────────────
# Hand-rolled validator (no jsonschema dep — matches Steps 04 / 05 / 06
# / 07 / 08 / 09). Checks:
#   - all 5 required top-level keys present
#   - goal_outcome.status in allowed enum, summary non-empty
#   - scenario_outcomes has exactly 3 entries with scenario_id 1/2/3 in order
#   - each scenario's title + predicted_resistance_type matches the
#     pre-mortem INPUT (defence-in-depth — runtime forces this)
#   - match_quality in allowed enum on every entry
#   - materialized is consistent with match_quality on every entry
#   - did_not_occur ⇒ empty evidence; direct_hit/partial ⇒ non-empty evidence
#   - notes non-empty on every entry
#   - unforeseen_moments kinds in allowed enum; optional turns in [1, N_USER_TURNS]
#   - prediction_accuracy in [0, 1]
#   - next_round_focus non-empty

schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
TOP_REQUIRED = set(schema["required"])

PREMORTEM_BY_ID = {
    s["scenario_id"]: s for s in PREMORTEM["failure_scenarios"]
}


def _is_non_empty_str(value) -> bool:
    return isinstance(value, str) and bool(value.strip())


def _materialized_consistent(outcome) -> bool:
    expected = outcome["match_quality"] != "did_not_occur"
    return bool(outcome["materialized"]) == expected


def _evidence_consistent(outcome) -> bool:
    if outcome["match_quality"] == "did_not_occur":
        return outcome["evidence"] == ""
    return _is_non_empty_str(outcome["evidence"])


scenario_outcomes = aftermath["scenario_outcomes"]

scenario_errors: list[str] = []
for i, outcome in enumerate(scenario_outcomes, start=1):
    expected_input = PREMORTEM_BY_ID.get(i)
    if outcome["scenario_id"] != i:
        scenario_errors.append(
            f"scenario {i}: scenario_id {outcome['scenario_id']!r} != {i}"
        )
    if expected_input is not None:
        expected_resistance = (
            expected_input["simulation_parameters"]["resistance_type"]
        )
        if outcome["predicted_resistance_type"] != expected_resistance:
            scenario_errors.append(
                f"scenario {i}: predicted_resistance_type "
                f"{outcome['predicted_resistance_type']!r} != "
                f"{expected_resistance!r} (forced from premortem input)"
            )
        if outcome["title"] != expected_input["title"]:
            scenario_errors.append(
                f"scenario {i}: title {outcome['title']!r} != input title "
                f"{expected_input['title']!r}"
            )
    if outcome["match_quality"] not in ALLOWED_MATCH_QUALITIES:
        scenario_errors.append(
            f"scenario {i}: match_quality {outcome['match_quality']!r} not in enum"
        )
    if not _materialized_consistent(outcome):
        scenario_errors.append(
            f"scenario {i}: materialized={outcome['materialized']!r} "
            f"inconsistent with match_quality={outcome['match_quality']!r}"
        )
    if not _evidence_consistent(outcome):
        scenario_errors.append(
            f"scenario {i}: evidence {outcome['evidence']!r} "
            f"inconsistent with match_quality={outcome['match_quality']!r}"
        )
    if not _is_non_empty_str(outcome["notes"]):
        scenario_errors.append(f"scenario {i}: notes empty or non-string")

unforeseen_errors: list[str] = []
for i, moment in enumerate(aftermath["unforeseen_moments"], start=1):
    if moment.get("kind") not in ALLOWED_UNFORESEEN_KINDS:
        unforeseen_errors.append(
            f"moment {i}: kind {moment.get('kind')!r} not in {ALLOWED_UNFORESEEN_KINDS}"
        )
    if not _is_non_empty_str(moment.get("description")):
        unforeseen_errors.append(f"moment {i}: description empty or non-string")
    if "turn" in moment and not (
        isinstance(moment["turn"], int) and 1 <= moment["turn"] <= N_USER_TURNS
    ):
        unforeseen_errors.append(
            f"moment {i}: turn {moment['turn']!r} not in [1, {N_USER_TURNS}]"
        )


checks: list[tuple[str, bool, str]] = [
    ("text_model_loaded",         True,                                                            f"{n_params:.1f}B params on {model.device}"),
    ("all_required_keys_present", TOP_REQUIRED.issubset(aftermath.keys()),                          f"{sorted(TOP_REQUIRED & set(aftermath.keys()))}"),
    ("goal_status_valid",         aftermath["goal_outcome"]["status"] in ALLOWED_GOAL_STATUSES,     aftermath["goal_outcome"]["status"]),
    ("goal_summary_non_empty",    _is_non_empty_str(aftermath["goal_outcome"]["summary"]),          aftermath["goal_outcome"]["summary"][:80] + "..."),
    ("scenario_count_exactly_3",  len(scenario_outcomes) == 3,                                      f"{len(scenario_outcomes)} entries"),
    ("scenario_ids_in_order",     [o["scenario_id"] for o in scenario_outcomes] == [1, 2, 3],       f"{[o['scenario_id'] for o in scenario_outcomes]}"),
    ("titles_match_premortem",    all(
                                       PREMORTEM_BY_ID[o["scenario_id"]]["title"] == o["title"]
                                       for o in scenario_outcomes
                                       if o["scenario_id"] in PREMORTEM_BY_ID
                                   ),                                                                f"3 scenarios"),
    ("resistance_matches_premortem", all(
                                          PREMORTEM_BY_ID[o["scenario_id"]]["simulation_parameters"]["resistance_type"]
                                          == o["predicted_resistance_type"]
                                          for o in scenario_outcomes
                                          if o["scenario_id"] in PREMORTEM_BY_ID
                                      ),                                                            f"3 scenarios"),
    ("scenarios_clean",           not scenario_errors,                                              f"{len(scenario_outcomes)} entries"),
    ("unforeseen_clean",          not unforeseen_errors,                                            f"{len(aftermath['unforeseen_moments'])} entries"),
    ("prediction_accuracy_unit",  0.0 <= aftermath["prediction_accuracy"] <= 1.0,                  f"{aftermath['prediction_accuracy']:.2f}"),
    ("next_round_focus_non_empty",_is_non_empty_str(aftermath["next_round_focus"]),                aftermath["next_round_focus"][:80] + "..."),
    ("aftermath_id_attached",     isinstance(aftermath.get("aftermath_id"), str)
                                   and aftermath["aftermath_id"].startswith("aftermath_"),         f"{aftermath.get('aftermath_id', '<missing>')}"),
]

print("=" * 76)
print("STEP 10 RESULTS — Gemma 4 Aftermath: Plan vs. Reality")
print("=" * 76)
all_ok = True
for name, ok, note in checks:
    icon = "PASS" if ok else "FAIL"
    print(f"[{icon}]  {name:32s}  {note}")
    if not ok:
        all_ok = False

if scenario_errors:
    print("\nscenario errors:")
    for line in scenario_errors:
        print(f"  - {line}")

if unforeseen_errors:
    print("\nunforeseen_moments errors:")
    for line in unforeseen_errors:
        print(f"  - {line}")

print()
print("OVERALL:", "READY FOR STEP 11" if all_ok else "FIX FAILURES ABOVE")

STEP 10 RESULTS — Gemma 4 Aftermath: Plan vs. Reality
[PASS]  text_model_loaded                 5.1B params on cuda:0
[PASS]  all_required_keys_present         ['goal_outcome', 'next_round_focus', 'prediction_accuracy', 'scenario_outcomes', 'unforeseen_moments']
[PASS]  goal_status_valid                 partial
[PASS]  goal_summary_non_empty            You got some good movement on the next steps, but Jamie still pushed back on the...
[PASS]  scenario_count_exactly_3          3 entries
[PASS]  scenario_ids_in_order             [1, 2, 3]
[PASS]  titles_match_premortem            3 scenarios
[PASS]  resistance_matches_premortem      3 scenarios
[PASS]  scenarios_clean                   3 entries
[PASS]  unforeseen_clean                  2 entries
[PASS]  prediction_accuracy_unit          0.75
[PASS]  next_round_focus_non_empty        Practice immediately following up any proposed solution with the specific commit...
[PASS]  aftermath_id_attached             aftermath_dad74fff3497

OVERAL

In [ ]:
# ── 8. A/B: enable_thinking=False vs enable_thinking=True (same inputs) ──
# Tests the open hypothesis [[hypothesis-persona-thinking-helps]] on a
# THIRD task family. Aftermath has the most cross-field structure yet:
#   - predicted resistance_type (per scenario, from pre-mortem)
#   - actual content in the transcript (per turn, from persona-sim)
#   - the user's response to that content (per turn, from user)
#   - a match_quality verdict that must be consistent with evidence
#   - a materialized boolean that must be consistent with match_quality
# Premortem (N=2) and debrief (N=1) both showed thinking-on tightening
# this kind of cross-field consistency. N=3 across two task families is
# the current state; this run is the third analytical-task datapoint.
#
# Token budget for the thinking branch — bumped to 4096 from the
# initial 2048 after the first Colab run truncated mid-trace. The
# aftermath input is the largest of the Phase 4 analytical tasks
# (rendered premortem block + debrief block + transcript all in one
# prompt), so the thinking trace expands proportionally — premortem
# only needed 2x its JSON body at 2048, aftermath needs more headroom.
# If 4096 also truncates, the finding is clean: aftermath is too
# token-hungry for the thinking channel at E2B scale, and
# thinking=False is the production default for this task family.
#
# Wraps the thinking call in try/except so a future failure prints the
# raw model output (attached to `exc.attempts`) without needing a re-run.

cfg_no_thinking = AftermathConfig(
    premortem=PREMORTEM,
    transcript=PRACTICE_TRANSCRIPT,
    user_goal=USER_GOAL,
    person_profile=JAMIE_PROFILE,
    debrief=DEBRIEF,
    enable_thinking=False,
)
cfg_thinking = AftermathConfig(
    premortem=PREMORTEM,
    transcript=PRACTICE_TRANSCRIPT,
    user_goal=USER_GOAL,
    person_profile=JAMIE_PROFILE,
    debrief=DEBRIEF,
    enable_thinking=True,
    max_new_tokens=4096,
)

after_no_thinking = generate_aftermath(processor, model, cfg=cfg_no_thinking)

after_thinking = None
thinking_err = None
try:
    after_thinking = generate_aftermath(processor, model, cfg=cfg_thinking)
except AftermathError as exc:
    thinking_err = exc


def _render_short(label: str, after: dict) -> None:
    print(f"\n--- {label} ---")
    print(f"  goal_outcome.status   : {after['goal_outcome']['status']}")
    print(f"  goal_outcome.summary  : {after['goal_outcome']['summary']}")
    for outcome in after["scenario_outcomes"]:
        print(
            f"  S{outcome['scenario_id']} "
            f"({outcome['predicted_resistance_type']}): "
            f"match_quality={outcome['match_quality']}, "
            f"materialized={outcome['materialized']}, "
            f"evidence={outcome['evidence']!r}"
        )
        print(f"      notes : {outcome['notes']}")
    print(f"  unforeseen_moments    : {len(after['unforeseen_moments'])}")
    for moment in after["unforeseen_moments"]:
        turn_str = f" @USER {moment['turn']}" if "turn" in moment else ""
        print(f"      [{moment['kind']}{turn_str}] {moment['description']}")
    print(f"  prediction_accuracy   : {after['prediction_accuracy']:.2f}")
    print(f"  next_round_focus      : {after['next_round_focus']}")


print("=" * 76)
print("A/B — enable_thinking on the same plan-vs-reality inputs")
print("=" * 76)

_render_short("thinking=False (default)", after_no_thinking)

if after_thinking is not None:
    _render_short("thinking=True", after_thinking)
else:
    print("\n--- thinking=True ---")
    print(f"  FAILED: {thinking_err}")
    for attempt in getattr(thinking_err, "attempts", []):
        print(f"\n  attempt ({attempt['sampling']}): {attempt['error']}")
        print("  raw_head (first 500 chars):")
        print(f"  {attempt['raw_head']!r}")

In [ ]:
# ── 9. Final aftermath dict — the JSON Phase 5's API will return verbatim ──
# Step 11 (Relationship Pulse) will consume one of these per practice
# round to track how prediction accuracy moves over time. Prefer the
# thinking=True aftermath when both branches succeeded (premortem's
# N=2 + debrief's N=1 finding was that thinking-on catches cross-field
# inconsistencies that thinking-off doesn't on analytical payloads);
# fall back to thinking=False otherwise. Contract test in cell 7
# already validated the structure; this cell is the demo handoff.

_source_aftermath = (
    after_thinking
    if ("after_thinking" in globals() and after_thinking is not None)
    else aftermath
)
_source_label = (
    "cell-8 thinking=True"
    if ("after_thinking" in globals() and after_thinking is not None)
    else "cell-6 thinking=False"
)

print("=" * 76)
print(f"FINAL AFTERMATH (source: {_source_label})")
print("=" * 76)
print(json.dumps(_source_aftermath, indent=2, ensure_ascii=False))